<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Declaraciones de exportación por región y aduana

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.

In [1]:
import pandas as pd
import numpy as np

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp

### Parámetros

Define las correcciones ortográficas y de tildes que estandarizan los nombres de las regiones
de procedencia.

In [3]:
correcciones = {
    'Bogota': 'Bogotá',
    'Valle Del Cauca': 'Valle del Cauca',
    'Bolivar': 'Bolívar',
    'Atlantico': 'Atlántico',
    'Norte De Santander': 'Norte de Santander',
    'Quindio': 'Quindío',
    'Boyaca': 'Boyacá',
    'Cordoba': 'Córdoba',
    'Departamentos Varios': 'Departamentos varios',
    'Choco': 'Chocó',
    'Caqueta': 'Caquetá',
    'San Andres': 'San Andrés',
    'Guainia': 'Guainía'
}

### Estandarización y agregación del formulario 600

Prepara la base de declaraciones de exportación.

**Pasos realizados:**

1. **Limpieza de columnas:** retira el sufijo del distrito capital de la región de procedencia, extrae el nombre de la aduana de despacho y deriva el código HS4 de la subpartida.

2. **Agregación:** cuenta los registros por combinación de aduana de despacho y región de procedencia.

3. **Normalización de regiones:** aplica las correcciones ortográficas y de tildes definidas en los parámetros.

4. **Etiqueta combinada:** crea la variable que une región y aduana para identificar los nodos logísticos.

In [4]:
df_600 = pd.read_parquet(raw / 'formulario600' / 'df_600.parquet')

df_600['REGION_PROCEDENCIA'] = df_600['REGION_PROCEDENCIA'].str.replace('DC','').str.title()
df_600['ADUANA_DESPACHO'] = df_600['ADUANA_DESPACHO'].map(lambda x: str(x).split('-')[0].strip().split(' de ')[-1])
df_600['HS4'] = df_600['SUBPARTIDA'].astype(str).str.zfill(10).str[:4]
df_600 = df_600.groupby(['ADUANA_DESPACHO','REGION_PROCEDENCIA'])[['FECHA_PROCESO']].count().sort_values(by='FECHA_PROCESO', ascending=False).rename(columns={'FECHA_PROCESO': 'CANTIDAD'}).reset_index()

df_600['REGION_PROCEDENCIA'] = (
    df_600['REGION_PROCEDENCIA']
    .str.strip()
    .str.title()
)

df_600['REGION_PROCEDENCIA'] = df_600['REGION_PROCEDENCIA'].replace(correcciones)

df_600['Label'] = df_600['REGION_PROCEDENCIA'] + ' - ' + df_600['ADUANA_DESPACHO']

temp.mkdir(parents=True, exist_ok=True)
df_600.to_parquet(temp / 'df_600.parquet', index=False)